# Testing with test dataset - January
The test plan in `Predict_analysis_plan.md`.
- Logistic Regression: with default setting (C = 1.0), fitted on all data spanning Nov 1st to Dec 24th, and then apply to the test dataset.
- All three must hold:
    - The flagged return at least two times as often as the average visitors.
    - The flagged purchase later at least four times of the average visitors.
    - Catch at least as many returners and later purchasers as the baseline rule.
- XGBoost: for comparison.

In [1]:
import pandas as pd
from sklearn.metrics import roc_auc_score

from ga4case.model import (
    OUTCOMES,
    load_first_visits,
    generate_input,
    rule_flag,
    logistic_model,
    xgboost_model,
    top_scored,
    flag_result,
)

In [2]:
def run_test(train_start, train_end, test_start, test_end):
    """Fit on arrivals in training dataset, score the test arrivals, and compare with the rule at the same number flagged"""
    train = load_first_visits(train_start, train_end)
    test = load_first_visits(test_start, test_end)
    X_train, X_test = generate_input(train), generate_input(test)
    rule = rule_flag(test)
    n_rule = int(rule.sum())

    rows = []
    for outcome in OUTCOMES:
        y = test[outcome].to_numpy()
        rows.append(
            {
                "outcome": outcome,
                "approach": "rule",
                "auc": roc_auc_score(y, rule),
                **flag_result(y, rule),
            }
        )
        for name, model in [
            ("logistic regression", logistic_model()),
            ("xgboost", xgboost_model()),
        ]:
            model.fit(X_train, train[outcome])
            score = model.predict_proba(X_test)[:, 1]
            rows.append(
                {
                    "outcome": outcome,
                    "approach": name,
                    "auc": roc_auc_score(y, score),
                    **flag_result(y, top_scored(score, n_rule)),
                }
            )
    return pd.DataFrame(rows)

In [3]:
# Check wether validation dataset reproduces results here
train_valid = run_test("2020-11-01", "2020-12-11", "2020-12-11", "2020-12-25")
train_valid.round(2)

,outcome,approach,auc,pct_average,pct_flagged,pct_outcome_among_flagged,times_average,cases_caught,pct_of_cases_caught
0,came_back,rule,0.59,8.23,18.0,15.98,1.94,1285,34.99
1,came_back,logistic regression,0.71,8.23,18.0,18.95,2.30,1523,41.46
2,came_back,xgboost,0.74,8.23,18.0,19.87,2.41,1597,43.48
3,purchase_later,rule,0.77,0.55,18.0,2.18,3.94,175,70.85
4,purchase_later,logistic regression,0.89,0.55,18.0,2.51,4.54,202,81.78
5,purchase_later,xgboost,0.91,0.55,18.0,2.54,4.59,204,82.59


In [4]:
# Overview two periods
def overview(periods):
    rows = []
    for label, start, end in periods:
        d = load_first_visits(start, end)
        rows.append(
            {
                "period": label,
                "users": len(d),
                "pct_flagged_rule": round(100 * rule_flag(d).mean(), 1),
                "pct_came_back": round(100 * d["came_back"].mean(), 1),
                "pct_purchase_later": round(100 * d["purchase_later"].mean(), 2),
            }
        )
    return pd.DataFrame(rows).set_index("period")


overview(
    [
        ("development: Nov 1 to Dec 24", "2020-11-01", "2020-12-25"),
        ("test: Jan 1 to Jan 24", "2021-01-01", "2021-01-25"),
    ]
)

,users,pct_flagged_rule,pct_came_back,pct_purchase_later
period,,,,
development: Nov 1 to Dec 24,152860,21.0,10.3,0.57
test: Jan 1 to Jan 24,70552,16.6,8.6,0.35


In [5]:
# Testing
january = run_test("2020-11-01", "2020-12-25", "2021-01-01", "2021-01-25")
january.round(2)

,outcome,approach,auc,pct_average,pct_flagged,pct_outcome_among_flagged,times_average,cases_caught,pct_of_cases_caught
0,came_back,rule,0.56,8.63,16.63,14.59,1.69,1712,28.12
1,came_back,logistic regression,0.66,8.63,16.63,17.91,2.08,2101,34.50
2,came_back,xgboost,0.71,8.63,16.63,18.86,2.19,2213,36.34
3,purchase_later,rule,0.78,0.35,16.63,1.52,4.39,178,72.95
4,purchase_later,logistic regression,0.88,0.35,16.63,1.68,4.86,197,80.74
5,purchase_later,xgboost,0.91,0.35,16.63,1.73,5.00,203,83.20


In [6]:
# Check whether testing pass the three holding standards
def check_pass(result):
    logit = result[result["approach"] == "logistic regression"].set_index("outcome")
    rule = result[result["approach"] == "rule"].set_index("outcome")
    checks = pd.DataFrame(
        {
            "result": [
                logit.loc["came_back", "times_average"],
                logit.loc["purchase_later", "times_average"],
                logit.loc["came_back", "cases_caught"]
                - rule.loc["came_back", "cases_caught"],
                logit.loc["purchase_later", "cases_caught"]
                - rule.loc["purchase_later", "cases_caught"],
            ],
            "needed_at_least": [2, 4, 0, 0],
        },
        index=[
            "returners: times the average",
            "later purchasers: times the average",
            "returners caught: model minus rule",
            "later purchasers caught: model minus rule",
        ],
    )
    checks["passed"] = checks["result"] >= checks["needed_at_least"]
    return checks


checks = check_pass(january)
print("All conditions hold:", bool(checks["passed"].all()))
checks.round(2)

All conditions hold: True


,result,needed_at_least,passed
returners: times the average,2.08,2,True
later purchasers: times the average,4.86,4,True
returners caught: model minus rule,389.00,0,True
later purchasers caught: model minus rule,19.00,0,True


Results:
- Test passed: all three hold standards were fulfilled.
- 70,552 new users in January and rule flagged 16.6%, with return rate 8.6% and later purchaser 0.35% -- both results lower than the outcomes from train/validation stage
- Returners: flagged return showed 2.08 times than the average, just slightly higher than the required 2 times with 2101 returners caught, and the rate lower than the validation outcome (2.30)
- Later purchasers: flagged outcome showed 4.86 times than the average with 197 purchasers caught. And the rate is similar to validation outcome: 80.74% vs 81.8%.
- Model caught 389 more returners and 19 more later purchasers compared with rule.
- XGBoost potentially could perform even better in catching returners and later purchasers here, as in comparison, it showed 2213 returners and 203 later purchaser, which were higher than Logistic regression. But this notebook still runs with Logistic regression as planned.
- This is still a predictive model not causal.


Notes: a run from a fresh clone showed that XGBoost comparison relies on the row oder of users table so sorting according to the user id was added to the `load_first_visits`. Only the XGBoost comparison shifted a little.